In [0]:
%sql
SHOW SCHEMAS IN wwi_source;

In [0]:
customers_df = spark.table("wwi_source.Sales.Customers")

print(f"Nombre de clients : {customers_df.count()}")

display(customers_df.limit(10))

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Tables Azure SQL nécessaires au projet
source_tables = {
    "customers": "wwi_source.Sales.Customers",
    "orders": "wwi_source.Sales.Orders",
    "order_lines": "wwi_source.Sales.OrderLines",
    "stock_items": "wwi_source.Warehouse.StockItems",
    "stock_item_holdings": "wwi_source.Warehouse.StockItemHoldings",
    "stock_item_transactions": "wwi_source.Warehouse.StockItemTransactions",
    "suppliers": "wwi_source.Purchasing.Suppliers",
    "purchase_orders": "wwi_source.Purchasing.PurchaseOrders",
    "purchase_order_lines": "wwi_source.Purchasing.PurchaseOrderLines"
}

ingestion_results = []

for target_name, source_table in source_tables.items():

    target_table = f"retail_dev.bronze.{target_name}"

    df = (
        spark.table(source_table)
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_system", lit("azure_sql_wwi"))
        .withColumn("_source_table", lit(source_table))
    )

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    row_count = spark.table(target_table).count()

    ingestion_results.append((target_table, row_count, "SUCCESS"))

    print(f"{target_table} : {row_count} lignes chargées")

In [0]:
results_df = spark.createDataFrame(
    ingestion_results,
    ["target_table", "row_count", "status"]
)

display(results_df)